# Ablation Study: Fine-Tuning emotion2vec+ base on UA-SER

Systematic ablation study for fine-tuning **emotion2vec+ base** (~94 M, data2vec architecture)
on the 4-class Ukrainian Speech Emotion Recognition dataset.


### Ablation factors (sequential / greedy)

1. **Freezing depth** — n_freeze in {2, 4, 6, 8}
2. **Loss function** — CE vs weighted CE vs focal
3. **Data augmentation** — on vs off
4. **Confidence weighting** — off vs 3-cue agreement

**Before running:** upload `data.zip` to Google Drive at `My Drive/thesis_new/data.zip` containing:
- `clips/` — 952 WAV files
- `dataset.csv`
- *(optional, for Exp 4)* `vad_results.csv`, `text_emotion_results.csv`, `text_emotion_llm_results.csv`

---
## 1. GPU Check & Install Dependencies

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    gpu_mem = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

!pip install -q funasr modelscope audiomentations[extras] librosa

---
## 2. Mount Google Drive & Unpack Data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import zipfile
from pathlib import Path

DRIVE_PROJECT = Path("/content/drive/MyDrive/thesis_new")
DATA_ZIP = DRIVE_PROJECT / "data.zip"
LOCAL_DATA = Path("/content/ser_data")

if not LOCAL_DATA.exists():
    assert DATA_ZIP.exists(), (
        f"Upload data.zip to Google Drive at: {DATA_ZIP}\n"
        f"The zip should contain: clips/, dataset.csv\n"
        f"Optionally: vad_results.csv, text_emotion_results.csv, text_emotion_llm_results.csv"
    )
    print(f"Extracting {DATA_ZIP} ...")
    with zipfile.ZipFile(DATA_ZIP, "r") as zf:
        zf.extractall(LOCAL_DATA)
    print("Done.")
else:
    print(f"Data already unpacked at {LOCAL_DATA}")

if (LOCAL_DATA / "dataset.csv").exists():
    DATA_ROOT = LOCAL_DATA
else:
    subdirs = [d for d in LOCAL_DATA.iterdir() if d.is_dir() and not d.name.startswith(".")]
    DATA_ROOT = subdirs[0] if subdirs else LOCAL_DATA

AUDIO_DIR = DATA_ROOT / "clips"
assert (DATA_ROOT / "dataset.csv").exists(), f"dataset.csv not found in {DATA_ROOT}"
assert AUDIO_DIR.exists(), f"clips/ not found in {DATA_ROOT}"

n_wavs = len(list(AUDIO_DIR.glob("*.wav")))
print(f"Data root:  {DATA_ROOT}")
print(f"Audio dir:  {AUDIO_DIR}  ({n_wavs} WAV files)")

---
## 3. Load Data & Build Splits

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

EMOTIONS = ["angry", "happy", "neutral", "sad"]
EMOTION2ID = {e: i for i, e in enumerate(EMOTIONS)}
ID2EMOTION = {i: e for i, e in enumerate(EMOTIONS)}
N_CLASSES = len(EMOTIONS)
SAMPLE_RATE = 16_000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df = pd.read_csv(DATA_ROOT / "dataset.csv")

train_df = df[df["split"] == "train"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

train_files    = train_df["filename"].tolist()
train_labels   = [EMOTION2ID[e] for e in train_df["emotion"]]
train_speakers = train_df["speaker_id"].values
test_files     = test_df["filename"].tolist()
test_labels    = [EMOTION2ID[e] for e in test_df["emotion"]]

class_counts = [sum(1 for l in train_labels if l == c) for c in range(N_CLASSES)]

missing = [f for f in train_files + test_files if not (AUDIO_DIR / f).exists()]
assert len(missing) == 0, f"{len(missing)} audio files missing, first: {missing[:5]}"

CUE_CSV_NAMES = ["vad_results.csv", "text_emotion_results.csv", "text_emotion_llm_results.csv"]
CUE_CSVS_PRESENT = all((DATA_ROOT / f).exists() for f in CUE_CSV_NAMES)

confidence_map = {}
AGREE_TO_WEIGHT = {0: 0.5, 1: 0.75, 2: 1.0, 3: 1.25}

if CUE_CSVS_PRESENT:
    vad = pd.read_csv(DATA_ROOT / "vad_results.csv")
    text_emo = pd.read_csv(DATA_ROOT / "text_emotion_results.csv")
    llm_emo = pd.read_csv(DATA_ROOT / "text_emotion_llm_results.csv")

    master = df.merge(
        vad[["filename", "valence", "arousal"]], on="filename", how="left"
    ).merge(
        text_emo[["filename", "text_pred_emotion"]], on="filename", how="left"
    ).merge(
        llm_emo[["filename", "llm_pred_emotion"]], on="filename", how="left"
    )

    centroids = master.groupby("emotion")[["valence", "arousal"]].mean()
    def vad_predict(row):
        dists = ((centroids - [row["valence"], row["arousal"]]) ** 2).sum(axis=1)
        return dists.idxmin()
    master["vad_pred_emotion"] = master.apply(vad_predict, axis=1)

    master["n_cue_agree"] = (
        (master["emotion"] == master["text_pred_emotion"]).astype(int)
        + (master["emotion"] == master["llm_pred_emotion"]).astype(int)
        + (master["emotion"] == master["vad_pred_emotion"]).astype(int)
    )
    confidence_map = dict(zip(master["filename"], master["n_cue_agree"].map(AGREE_TO_WEIGHT)))
    print(f"Confidence weights: AVAILABLE ({master['n_cue_agree'].value_counts().sort_index().to_dict()})")
elif "n_agree" in df.columns:
    confidence_map = dict(zip(df["filename"], df["n_agree"].map(AGREE_TO_WEIGHT)))
    print("Confidence weights: AVAILABLE (from n_agree column)")
else:
    print("Confidence weights: NOT AVAILABLE (cue CSVs not in zip, Exp 4 will be skipped)")

print(f"\nTrain: {len(train_files)} clips  |  Test: {len(test_files)} clips (speaker-disjoint)")
print(f"Classes: {dict(zip(EMOTIONS, class_counts))}")
print(f"Device: {DEVICE}")

---
## 4. Load emotion2vec Encoder Weights

In [ ]:
from funasr import AutoModel as FunASRAutoModel

print("Downloading emotion2vec+ base ...")
_funasr_wrapper = FunASRAutoModel(model="iic/emotion2vec_plus_base")
ENCODER_WEIGHTS = _funasr_wrapper.model.state_dict()
ENCODER_CFG = _funasr_wrapper.model.cfg

EMBED_DIM = ENCODER_CFG.get("embed_dim")
N_BLOCKS = ENCODER_CFG.get("depth")
NORMALIZE_INPUT = ENCODER_CFG.get("normalize", True)

print(f"  embed_dim  = {EMBED_DIM}")
print(f"  depth      = {N_BLOCKS} transformer blocks")
print(f"  normalize  = {NORMALIZE_INPUT}")
print(f"  Total params: {sum(p.numel() for p in _funasr_wrapper.model.parameters()) / 1e6:.1f} M")

del _funasr_wrapper
torch.cuda.empty_cache()
print("Encoder weights cached. FunASR wrapper released.")

---
## 5. Model, Dataset & Loss Definitions

In [ ]:
import random
import math
import librosa
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from funasr.models.emotion2vec.model import Emotion2vec as Emotion2vecModule
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    cohen_kappa_score, confusion_matrix, f1_score,
)
from tqdm.auto import tqdm

class SERDataset(Dataset):
    def __init__(self, filenames, labels, audio_dir, max_duration=6.0,
                 augment=None, sample_weights=None, is_train=False):
        self.filenames = filenames
        self.labels = labels
        self.audio_dir = Path(audio_dir)
        self.max_samples = int(max_duration * SAMPLE_RATE)
        self.augment = augment
        self.sample_weights = sample_weights or [1.0] * len(filenames)
        self.is_train = is_train

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        path = self.audio_dir / self.filenames[idx]
        wav, _ = librosa.load(str(path), sr=SAMPLE_RATE, mono=True)

        if len(wav) > self.max_samples:
            if self.is_train:
                start = random.randint(0, len(wav) - self.max_samples)
            else:
                start = (len(wav) - self.max_samples) // 2
            wav = wav[start : start + self.max_samples]

        if self.augment is not None:
            wav = self.augment(samples=wav, sample_rate=SAMPLE_RATE)
            if len(wav) > self.max_samples:
                wav = wav[: self.max_samples]

        wav_t = torch.tensor(wav, dtype=torch.float32)
        if NORMALIZE_INPUT:
            wav_t = F.layer_norm(wav_t, wav_t.shape)

        return wav_t, self.labels[idx], self.sample_weights[idx]

def collate_fn(batch):
    wavs, labels, weights = zip(*batch)
    max_len = max(w.shape[0] for w in wavs)
    padded = torch.zeros(len(wavs), max_len)
    pad_mask = torch.ones(len(wavs), max_len, dtype=torch.bool)
    for i, w in enumerate(wavs):
        padded[i, : w.shape[0]] = w
        pad_mask[i, : w.shape[0]] = False
    return (
        padded, pad_mask,
        torch.tensor(labels, dtype=torch.long),
        torch.tensor(weights, dtype=torch.float32),
    )

def _build_emotion2vec_encoder(encoder_cfg, encoder_weights):
    from omegaconf import OmegaConf
    cfg_dict = OmegaConf.to_container(encoder_cfg, resolve=True)
    model = Emotion2vecModule(model_conf=cfg_dict, vocab_size=9)
    model.load_state_dict(encoder_weights, strict=True)
    return model

class Emotion2vecSER(nn.Module):
    def __init__(self, encoder_cfg, encoder_weights,
                 n_classes=4, n_freeze=6, dropout=0.3):
        super().__init__()
        self.encoder = _build_emotion2vec_encoder(encoder_cfg, encoder_weights)
        embed_dim = encoder_cfg.get("embed_dim")
        n_blocks = encoder_cfg.get("depth")

        self.encoder.proj = None

        for param in self.encoder.modality_encoders.parameters():
            param.requires_grad = False

        n_freeze = min(n_freeze, n_blocks)
        for i, block in enumerate(self.encoder.blocks):
            for param in block.parameters():
                param.requires_grad = i >= n_freeze

        if self.encoder.dropout_input is not None:
            for param in self.encoder.dropout_input.parameters():
                param.requires_grad = False
        if self.encoder.norm is not None:
            for param in self.encoder.norm.parameters():
                param.requires_grad = True

        self.attention = nn.Linear(embed_dim, 1)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, n_classes),
        )

        n_trainable = sum(1 for i in range(n_blocks) if i >= n_freeze)
        total_p = sum(p.numel() for p in self.parameters())
        train_p = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(
            f"  Encoder: {n_blocks} blocks \u2014 freezing bottom {n_freeze}, "
            f"fine-tuning top {n_trainable}"
        )
        print(f"  Params: {total_p/1e6:.1f} M total, {train_p/1e6:.1f} M trainable")

    def forward(self, input_values, padding_mask=None):
        feats = self.encoder.extract_features(
            source=input_values, padding_mask=padding_mask,
            mask=False, remove_extra_tokens=True,
        )
        hidden = feats["x"]
        feat_mask = feats["padding_mask"]

        attn_weights = self.attention(hidden)
        if feat_mask is not None:
            attn_weights = attn_weights.masked_fill(feat_mask.unsqueeze(-1), float("-inf"))
        attn_weights = torch.softmax(attn_weights, dim=1)

        pooled = (hidden * attn_weights).sum(dim=1)
        return self.classifier(pooled)

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        return (1 - pt) ** self.gamma * ce

def build_loss(loss_type, class_counts, device):
    weights = 1.0 / torch.tensor(class_counts, dtype=torch.float32)
    weights = weights / weights.sum() * len(weights)
    weights = weights.to(device)
    if loss_type == "ce":
        return nn.CrossEntropyLoss(reduction="none")
    if loss_type == "weighted_ce":
        return nn.CrossEntropyLoss(weight=weights, reduction="none")
    if loss_type == "focal":
        return FocalLoss(gamma=2.0, weight=weights)
    raise ValueError(f"Unknown loss: {loss_type}")

def build_augmentation():
    try:
        import audiomentations as A
    except ImportError:
        print("WARNING: audiomentations not installed.")
        return None
    return A.Compose([
        A.AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.4),
        A.TimeStretch(min_rate=0.85, max_rate=1.15, p=0.4),
        A.PitchShift(min_semitones=-2, max_semitones=2, p=0.4),
        A.Gain(min_gain_db=-6, max_gain_db=6, p=0.3),
        A.AddGaussianSNR(min_snr_db=15, max_snr_db=40, p=0.3),
    ])

def train_one_epoch(model, loader, optimizer, scheduler, criterion, device, grad_accum, scaler):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []
    use_amp = scaler.is_enabled()

    optimizer.zero_grad()
    for step, (wavs, masks, labels, sample_w) in enumerate(loader):
        wavs, masks = wavs.to(device), masks.to(device)
        labels, sample_w = labels.to(device), sample_w.to(device)

        with torch.amp.autocast("cuda", enabled=use_amp):
            logits = model(wavs, padding_mask=masks)
            per_sample_loss = criterion(logits, labels)
            loss = (per_sample_loss * sample_w).mean() / grad_accum

        scaler.scale(loss).backward()

        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            if scheduler is not None:
                scheduler.step()

        with torch.no_grad():
            total_loss += per_sample_loss.mean().item() * labels.size(0)
        all_preds.extend(logits.argmax(-1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    return total_loss / len(all_labels), balanced_accuracy_score(all_labels, all_preds)

@torch.no_grad()
def evaluate_model(model, loader, criterion, device, use_amp=False):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for wavs, masks, labels, _ in loader:
        wavs, masks, labels = wavs.to(device), masks.to(device), labels.to(device)
        with torch.amp.autocast("cuda", enabled=use_amp and device.type == "cuda"):
            logits = model(wavs, padding_mask=masks)
            loss = criterion(logits, labels).mean()
        total_loss += loss.item() * labels.size(0)
        all_preds.extend(logits.argmax(-1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    avg_loss = total_loss / len(all_labels)
    uar = balanced_accuracy_score(all_labels, all_preds)
    return avg_loss, uar, all_preds, all_labels

def compute_all_metrics(y_true, y_pred):
    per_class = {}
    report = classification_report(
        y_true, y_pred, target_names=EMOTIONS,
        labels=list(range(N_CLASSES)), output_dict=True, zero_division=0,
    )
    for emo in EMOTIONS:
        per_class[emo] = report[emo]["recall"]
    return {
        "UAR": balanced_accuracy_score(y_true, y_pred),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro F1": f1_score(y_true, y_pred, average="macro", labels=list(range(N_CLASSES))),
        **{f"{emo.capitalize()} R": per_class[emo] for emo in EMOTIONS},
    }

print("All definitions loaded.")

---
## 6. Plotting Helpers

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

sns.set_theme(style="whitegrid", font_scale=1.2)
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 300,
    "font.family": "serif", "axes.titlesize": 14, "axes.labelsize": 12,
})

def plot_confusion_matrix(y_true, y_pred, title, save_path=None):
    y_true_n = [ID2EMOTION[y] for y in y_true]
    y_pred_n = [ID2EMOTION[y] for y in y_pred]
    cm = confusion_matrix(y_true_n, y_pred_n, labels=EMOTIONS)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(cm, annot=True, fmt="d", ax=axes[0], cmap="Blues",
                xticklabels=EMOTIONS, yticklabels=EMOTIONS)
    axes[0].set_title("Raw Counts")
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

    sns.heatmap(cm_norm, annot=True, fmt=".2f", ax=axes[1], cmap="Blues",
                xticklabels=EMOTIONS, yticklabels=EMOTIONS, vmin=0, vmax=1)
    axes[1].set_title("Normalized (row = recall)")
    axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")

    fig.suptitle(title, fontsize=15, y=1.02)
    fig.tight_layout()
    if save_path:
        for ext in ("png", "pdf"):
            fig.savefig(f"{save_path}.{ext}", bbox_inches="tight")
    plt.show()
    plt.close(fig)

def plot_training_curves(hist, title="", save_path=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(hist["train_loss"], label="Train", linewidth=2)
    ax1.plot(hist["val_loss"], label="Validation", linewidth=2)
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Loss")
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(hist["train_uar"], label="Train", linewidth=2)
    ax2.plot(hist["val_uar"], label="Validation", linewidth=2)
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("UAR"); ax2.set_title("UAR")
    ax2.legend(); ax2.grid(True, alpha=0.3)

    if title:
        fig.suptitle(title, fontsize=14, y=1.02)
    fig.tight_layout()
    if save_path:
        for ext in ("png", "pdf"):
            fig.savefig(f"{save_path}.{ext}", bbox_inches="tight")
    plt.show()
    plt.close(fig)

print("Plotting helpers ready.")

---
## 7. `run_experiment()` — Single-Run Wrapper

Wraps the entire train → evaluate pipeline into a reusable function.
Returns metrics dict, predictions, training history, and best checkpoint.

In [ ]:
def run_experiment(cfg: dict, verbose: bool = True) -> dict:
    import gc
    torch.manual_seed(cfg.get("seed", 42))
    np.random.seed(cfg.get("seed", 42))
    random.seed(cfg.get("seed", 42))

    run_name = cfg["run_name"]
    if verbose:
        print(f"\n{'=' * 70}")
        print(f"  {run_name}")
        key_params = {k: cfg[k] for k in ["n_freeze", "loss", "augment", "confidence_weighting"]}
        print(f"  {key_params}")
        print(f"{'=' * 70}")

    splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=cfg.get("seed", 42))
    tr_idx, es_idx = next(splitter.split(train_files, train_labels, groups=train_speakers))

    ft_train_files   = [train_files[i] for i in tr_idx]
    ft_train_labels  = [train_labels[i] for i in tr_idx]
    ft_val_files     = [train_files[i] for i in es_idx]
    ft_val_labels    = [train_labels[i] for i in es_idx]

    use_conf = cfg.get("confidence_weighting", False) and bool(confidence_map)
    ft_train_weights = (
        [confidence_map.get(f, 1.0) for f in ft_train_files] if use_conf
        else [1.0] * len(ft_train_files)
    )

    augment = build_augmentation() if cfg.get("augment", True) else None

    train_ds = SERDataset(
        ft_train_files, ft_train_labels, str(AUDIO_DIR),
        max_duration=cfg.get("max_duration", 6.0), augment=augment,
        sample_weights=ft_train_weights, is_train=True,
    )
    val_ds = SERDataset(
        ft_val_files, ft_val_labels, str(AUDIO_DIR),
        max_duration=cfg.get("max_duration", 6.0), is_train=False,
    )

    batch_size = cfg.get("batch_size", 16)
    grad_accum = cfg.get("grad_accum_steps", 2)
    use_pin = DEVICE.type == "cuda"
    num_workers = cfg.get("num_workers", 0)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        collate_fn=collate_fn, num_workers=num_workers, pin_memory=use_pin,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        collate_fn=collate_fn, num_workers=num_workers, pin_memory=use_pin,
    )

    model = Emotion2vecSER(
        ENCODER_CFG, ENCODER_WEIGHTS,
        n_classes=N_CLASSES,
        n_freeze=cfg.get("n_freeze", 4),
        dropout=cfg.get("dropout", 0.4),
    ).to(DEVICE)

    head_params = list(model.attention.parameters()) + list(model.classifier.parameters())
    head_ids = {id(p) for p in head_params}
    backbone_params = [
        p for p in model.parameters() if p.requires_grad and id(p) not in head_ids
    ]

    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": cfg.get("backbone_lr", 5e-5)},
            {"params": head_params,     "lr": cfg.get("head_lr", 5e-4)},
        ],
        weight_decay=cfg.get("weight_decay", 0.01),
    )

    max_epochs = cfg.get("max_epochs", 50)
    steps_per_epoch = math.ceil(len(train_loader) / grad_accum)
    total_steps = steps_per_epoch * max_epochs
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[cfg.get("backbone_lr", 5e-5), cfg.get("head_lr", 5e-4)],
        total_steps=total_steps,
        pct_start=cfg.get("warmup_ratio", 0.1),
        anneal_strategy="cos",
    )

    criterion = build_loss(cfg.get("loss", "weighted_ce"), class_counts, DEVICE)
    use_amp = cfg.get("use_amp", True) and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    patience = cfg.get("patience", 12)
    best_uar = 0.0
    best_state = None
    patience_ctr = 0
    hist = {"train_loss": [], "val_loss": [], "train_uar": [], "val_uar": []}

    for epoch in range(max_epochs):
        tr_loss, tr_uar = train_one_epoch(
            model, train_loader, optimizer, scheduler,
            criterion, DEVICE, grad_accum, scaler,
        )
        va_loss, va_uar, _, _ = evaluate_model(
            model, val_loader, criterion, DEVICE, use_amp=use_amp,
        )

        hist["train_loss"].append(tr_loss)
        hist["val_loss"].append(va_loss)
        hist["train_uar"].append(tr_uar)
        hist["val_uar"].append(va_uar)

        improved = va_uar > best_uar
        if improved:
            best_uar = va_uar
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1

        if verbose:
            star = " *" if improved else ""
            print(
                f"  Epoch {epoch+1:3d}/{max_epochs}  "
                f"TrLoss={tr_loss:.4f} TrUAR={tr_uar:.3f}  |  "
                f"VaLoss={va_loss:.4f} VaUAR={va_uar:.3f}{star}"
            )

        if patience_ctr >= patience:
            if verbose:
                print(f"  Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)
    test_ds = SERDataset(
        test_files, test_labels, str(AUDIO_DIR),
        max_duration=cfg.get("max_duration", 6.0), is_train=False,
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        collate_fn=collate_fn, num_workers=num_workers, pin_memory=use_pin,
    )

    _, _, test_preds, test_labels_out = evaluate_model(
        model, test_loader, criterion, DEVICE, use_amp=use_amp,
    )
    test_metrics = compute_all_metrics(test_labels_out, test_preds)

    if verbose:
        print(f"\n  TEST  UAR={test_metrics['UAR']:.3f}  "
              f"Acc={test_metrics['Accuracy']:.3f}  "
              f"F1={test_metrics['Macro F1']:.3f}  "
)

    del model, optimizer, scheduler, scaler, criterion
    del train_loader, val_loader, test_loader
    del train_ds, val_ds, test_ds
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "run_name": run_name,
        "cfg": cfg,
        "metrics": test_metrics,
        "test_preds": test_preds,
        "test_labels": test_labels_out,
        "hist": hist,
        "best_state": best_state,
        "best_val_uar": best_uar,
        "epochs_trained": len(hist["train_loss"]),
    }

print("run_experiment() ready.")

---
## 8. Ablation Experiment Configs

**Sequential / greedy design:**
1. Sweep freezing depth (most impactful) → pick best
2. Sweep loss function with best freeze → pick best
3. Sweep augmentation with best freeze+loss → pick best
4. Sweep confidence weighting with best config → final best

In [ ]:
SHARED_CFG = dict(
    dropout=0.4,
    head_lr=5e-4,
    backbone_lr=5e-5,
    weight_decay=0.01,
    batch_size=16,
    grad_accum_steps=2,
    max_epochs=50,
    warmup_ratio=0.1,
    patience=12,
    use_amp=True,
    max_duration=6.0,
    seed=42,
    num_workers=0,
)

EXP1_CONFIGS = []
for nf in [8, 6, 4, 2]:
    EXP1_CONFIGS.append({
        **SHARED_CFG,
        "run_name": f"exp1_freeze{nf}",
        "experiment": "1_freezing",
        "n_freeze": nf,
        "loss": "weighted_ce",
        "augment": True,
        "confidence_weighting": False,
    })

print(f"Experiment 1: {len(EXP1_CONFIGS)} runs (freezing depth)")
for c in EXP1_CONFIGS:
    print(f"  {c['run_name']:25s}  n_freeze={c['n_freeze']}")

---
## 9. Run Experiment 1 — Freezing Depth

In [ ]:
from datetime import datetime

ALL_RESULTS = []

print(f"Starting Experiment 1: Freezing Depth ({len(EXP1_CONFIGS)} runs)")
print(f"Time: {datetime.now().strftime('%H:%M:%S')}")

for cfg in EXP1_CONFIGS:
    result = run_experiment(cfg)
    ALL_RESULTS.append(result)

exp1_results = [r for r in ALL_RESULTS if r["cfg"]["experiment"] == "1_freezing"]
best_exp1 = max(exp1_results, key=lambda r: r["metrics"]["UAR"])
BEST_FREEZE = best_exp1["cfg"]["n_freeze"]

print(f"\n{'=' * 70}")
print(f"  EXPERIMENT 1 SUMMARY")
print(f"{'=' * 70}")
for r in exp1_results:
    m = r["metrics"]
    tag = " << BEST" if r["cfg"]["n_freeze"] == BEST_FREEZE else ""
    print(f"  freeze={r['cfg']['n_freeze']}  UAR={m['UAR']:.3f}  Acc={m['Accuracy']:.3f}  "
          f"F1={m['Macro F1']:.3f}  {tag}")
print(f"\n  Best freezing depth: n_freeze={BEST_FREEZE}")
print(f"  Time: {datetime.now().strftime('%H:%M:%S')}")

---
## 10. Run Experiment 2 — Loss Function

In [ ]:
EXP2_CONFIGS = []
for loss_type in ["ce", "weighted_ce", "focal"]:
    EXP2_CONFIGS.append({
        **SHARED_CFG,
        "run_name": f"exp2_{loss_type}",
        "experiment": "2_loss",
        "n_freeze": BEST_FREEZE,
        "loss": loss_type,
        "augment": True,
        "confidence_weighting": False,
    })

print(f"Starting Experiment 2: Loss Function (using n_freeze={BEST_FREEZE})")
for cfg in EXP2_CONFIGS:
    exp1_match = next(
        (r for r in ALL_RESULTS
         if r["cfg"]["n_freeze"] == cfg["n_freeze"]
         and r["cfg"]["loss"] == cfg["loss"]
         and r["cfg"]["augment"] == cfg["augment"]
         and r["cfg"]["confidence_weighting"] == cfg["confidence_weighting"]
         and r["cfg"]["experiment"] == "1_freezing"),
        None,
    )
    if exp1_match:
        print(f"  Reusing {exp1_match['run_name']} for {cfg['run_name']}")
        reused = {**exp1_match, "run_name": cfg["run_name"], "cfg": cfg}
        ALL_RESULTS.append(reused)
    else:
        result = run_experiment(cfg)
        ALL_RESULTS.append(result)

exp2_results = [r for r in ALL_RESULTS if r["cfg"]["experiment"] == "2_loss"]
best_exp2 = max(exp2_results, key=lambda r: r["metrics"]["UAR"])
BEST_LOSS = best_exp2["cfg"]["loss"]

print(f"\n{'=' * 70}")
print(f"  EXPERIMENT 2 SUMMARY (n_freeze={BEST_FREEZE})")
print(f"{'=' * 70}")
for r in exp2_results:
    m = r["metrics"]
    tag = " << BEST" if r["cfg"]["loss"] == BEST_LOSS else ""
    print(f"  loss={r['cfg']['loss']:12s}  UAR={m['UAR']:.3f}  Acc={m['Accuracy']:.3f}  "
          f"F1={m['Macro F1']:.3f}  {tag}")
print(f"\n  Best loss: {BEST_LOSS}")

---
## 11. Run Experiment 3 — Data Augmentation

In [ ]:
EXP3_CONFIGS = []
for aug in [True, False]:
    EXP3_CONFIGS.append({
        **SHARED_CFG,
        "run_name": f"exp3_aug{'_on' if aug else '_off'}",
        "experiment": "3_augmentation",
        "n_freeze": BEST_FREEZE,
        "loss": BEST_LOSS,
        "augment": aug,
        "confidence_weighting": False,
    })

print(f"Starting Experiment 3: Augmentation (n_freeze={BEST_FREEZE}, loss={BEST_LOSS})")
for cfg in EXP3_CONFIGS:
    prev_match = next(
        (r for r in ALL_RESULTS
         if r["cfg"]["n_freeze"] == cfg["n_freeze"]
         and r["cfg"]["loss"] == cfg["loss"]
         and r["cfg"]["augment"] == cfg["augment"]
         and r["cfg"]["confidence_weighting"] == cfg["confidence_weighting"]
         and r["cfg"]["experiment"] in ("1_freezing", "2_loss")),
        None,
    )
    if prev_match:
        print(f"  Reusing {prev_match['run_name']} for {cfg['run_name']}")
        reused = {**prev_match, "run_name": cfg["run_name"], "cfg": cfg}
        ALL_RESULTS.append(reused)
    else:
        result = run_experiment(cfg)
        ALL_RESULTS.append(result)

exp3_results = [r for r in ALL_RESULTS if r["cfg"]["experiment"] == "3_augmentation"]
best_exp3 = max(exp3_results, key=lambda r: r["metrics"]["UAR"])
BEST_AUG = best_exp3["cfg"]["augment"]

print(f"\n{'=' * 70}")
print(f"  EXPERIMENT 3 SUMMARY (n_freeze={BEST_FREEZE}, loss={BEST_LOSS})")
print(f"{'=' * 70}")
for r in exp3_results:
    m = r["metrics"]
    tag = " << BEST" if r["cfg"]["augment"] == BEST_AUG else ""
    print(f"  augment={str(r['cfg']['augment']):5s}  UAR={m['UAR']:.3f}  Acc={m['Accuracy']:.3f}  "
          f"F1={m['Macro F1']:.3f}  {tag}")
print(f"\n  Best augmentation: {BEST_AUG}")

---
## 12. Run Experiment 4 — Confidence Weighting

In [ ]:
EXP4_CONFIGS = []
conf_options = [False, True] if CUE_CSVS_PRESENT else [False]
for cw in conf_options:
    EXP4_CONFIGS.append({
        **SHARED_CFG,
        "run_name": f"exp4_conf{'_on' if cw else '_off'}",
        "experiment": "4_confidence",
        "n_freeze": BEST_FREEZE,
        "loss": BEST_LOSS,
        "augment": BEST_AUG,
        "confidence_weighting": cw,
    })

if not CUE_CSVS_PRESENT:
    print("Cue CSVs not found in data.zip \u2014 skipping confidence weighting (only running OFF).")

print(f"Starting Experiment 4: Confidence Weighting "
      f"(n_freeze={BEST_FREEZE}, loss={BEST_LOSS}, augment={BEST_AUG})")

for cfg in EXP4_CONFIGS:
    prev_match = next(
        (r for r in ALL_RESULTS
         if r["cfg"]["n_freeze"] == cfg["n_freeze"]
         and r["cfg"]["loss"] == cfg["loss"]
         and r["cfg"]["augment"] == cfg["augment"]
         and r["cfg"]["confidence_weighting"] == cfg["confidence_weighting"]
         and r["cfg"]["experiment"] in ("1_freezing", "2_loss", "3_augmentation")),
        None,
    )
    if prev_match:
        print(f"  Reusing {prev_match['run_name']} for {cfg['run_name']}")
        reused = {**prev_match, "run_name": cfg["run_name"], "cfg": cfg}
        ALL_RESULTS.append(reused)
    else:
        result = run_experiment(cfg)
        ALL_RESULTS.append(result)

exp4_results = [r for r in ALL_RESULTS if r["cfg"]["experiment"] == "4_confidence"]
best_exp4 = max(exp4_results, key=lambda r: r["metrics"]["UAR"])
BEST_CONF = best_exp4["cfg"]["confidence_weighting"]

print(f"\n{'=' * 70}")
print(f"  EXPERIMENT 4 SUMMARY")
print(f"{'=' * 70}")
for r in exp4_results:
    m = r["metrics"]
    tag = " << BEST" if r["cfg"]["confidence_weighting"] == BEST_CONF else ""
    print(f"  conf_wt={str(r['cfg']['confidence_weighting']):5s}  UAR={m['UAR']:.3f}  "
          f"Acc={m['Accuracy']:.3f}  F1={m['Macro F1']:.3f}  {tag}")

print(f"\n  FINAL BEST CONFIG:")
print(f"    n_freeze             = {BEST_FREEZE}")
print(f"    loss                 = {BEST_LOSS}")
print(f"    augment              = {BEST_AUG}")
print(f"    confidence_weighting = {BEST_CONF}")
print(f"\n  Time: {datetime.now().strftime('%H:%M:%S')}")

---
## 13. Results Dashboard

Comprehensive visualisation of all ablation runs.

In [ ]:
from IPython.display import display, Markdown

rows = []
seen_configs = set()
for r in ALL_RESULTS:
    cfg_key = (r["cfg"]["n_freeze"], r["cfg"]["loss"],
               r["cfg"]["augment"], r["cfg"]["confidence_weighting"])
    if cfg_key in seen_configs:
        continue
    seen_configs.add(cfg_key)
    rows.append({
        "Run": r["run_name"],
        "Experiment": r["cfg"]["experiment"],
        "n_freeze": r["cfg"]["n_freeze"],
        "Loss": r["cfg"]["loss"],
        "Augment": r["cfg"]["augment"],
        "Conf. Wt.": r["cfg"]["confidence_weighting"],
        "Epochs": r["epochs_trained"],
        "Val UAR": r["best_val_uar"],
        **r["metrics"],
    })

results_df = pd.DataFrame(rows)
display(Markdown("### Full Ablation Results"))

fmt_df = results_df.copy()
pct_cols = ["UAR", "Accuracy", "Macro F1", "Val UAR",
            "Angry R", "Happy R", "Neutral R", "Sad R"]
for c in pct_cols:
    if c in fmt_df.columns:
        fmt_df[c] = fmt_df[c].apply(lambda x: f"{x:.1%}")

display(fmt_df.style.set_properties(**{"text-align": "center", "font-size": "11px"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "center"), ("font-weight", "bold")]}]
))

In [ ]:
display(Markdown("### Experiment 1: Effect of Freezing Depth"))

exp1_df = results_df[results_df["Experiment"] == "1_freezing"].sort_values("n_freeze")
fig, ax = plt.subplots(figsize=(10, 5))

x = range(len(exp1_df))
labels_x = [f"freeze {nf}\n(train {8-nf})" for nf in exp1_df["n_freeze"]]

for metric, color, marker in [("UAR", "#2ecc71", "o"), ("Macro F1", "#3498db", "s"),
                               ("Accuracy", "#e67e22", "^")]:
    ax.plot(x, exp1_df[metric].values, f"-{marker}", label=metric, color=color,
            linewidth=2, markersize=8)

ax.axhline(y=0.539, color="gray", linestyle="--", alpha=0.6, label="Zero-shot UAR (53.9%)")
ax.axhline(y=0.25, color="red", linestyle=":", alpha=0.5, label="Chance (25%)")
ax.set_xticks(x)
ax.set_xticklabels(labels_x)
ax.set_ylabel("Score")
ax.set_title("Effect of Freezing Depth on Test Performance")
ax.legend(loc="best")
ax.set_ylim(0.15, 0.85)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
display(Markdown("### Experiment 2: Effect of Loss Function"))

exp2_df = results_df[results_df["Experiment"] == "2_loss"]
fig, ax = plt.subplots(figsize=(8, 5))

metrics_to_plot = ["UAR", "Macro F1", "Accuracy"]
x = np.arange(len(metrics_to_plot))
width = 0.22
colors = {"ce": "#e74c3c", "weighted_ce": "#3498db", "focal": "#2ecc71"}

for i, (_, row) in enumerate(exp2_df.iterrows()):
    vals = [row[m] for m in metrics_to_plot]
    bars = ax.bar(x + i * width, vals, width, label=row["Loss"],
                  color=colors[row["Loss"]], edgecolor="white")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{v:.1%}", ha="center", va="bottom", fontsize=8, fontweight="bold")

ax.axhline(y=0.25, color="gray", linestyle="--", alpha=0.5)
ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylabel("Score")
ax.set_title(f"Loss Function Comparison (n_freeze={BEST_FREEZE})")
ax.legend(title="Loss")
ax.set_ylim(0, 0.85)
fig.tight_layout()
plt.show()

In [ ]:
display(Markdown("### Per-Class Recall Across All Runs"))

recall_cols = [f"{e.capitalize()} R" for e in EMOTIONS]
heatmap_df = results_df.set_index("Run")[recall_cols].copy()
heatmap_df.columns = [e.capitalize() for e in EMOTIONS]

fig, ax = plt.subplots(figsize=(10, max(3, len(heatmap_df) * 0.5 + 1)))
sns.heatmap(heatmap_df, annot=True, fmt=".2f", cmap="YlOrRd",
            vmin=0, vmax=1, linewidths=0.5, ax=ax,
            cbar_kws={"shrink": 0.8})
ax.set_title("Per-Class Recall by Configuration")
ax.set_ylabel("")
fig.tight_layout()
plt.show()

In [ ]:
display(Markdown("### Best Model: Confusion Matrix"))

best_overall = max(ALL_RESULTS, key=lambda r: r["metrics"]["UAR"])
best_m = best_overall["metrics"]
print(f"Best run: {best_overall['run_name']}")
print(f"  UAR={best_m['UAR']:.3f}  Acc={best_m['Accuracy']:.3f}  "
      f"F1={best_m['Macro F1']:.3f}")

plot_confusion_matrix(
    best_overall["test_labels"], best_overall["test_preds"],
    f"Best Config: {best_overall['run_name']}",
)

print("\nClassification Report:")
print(classification_report(
    [ID2EMOTION[y] for y in best_overall["test_labels"]],
    [ID2EMOTION[y] for y in best_overall["test_preds"]],
    target_names=EMOTIONS, labels=EMOTIONS, digits=3,
))

In [ ]:
display(Markdown("### Best Model: Training Curves"))

plot_training_curves(
    best_overall["hist"],
    title=f"Training Curves: {best_overall['run_name']}",
)

In [ ]:
display(Markdown("### Factor Impact Summary"))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

exp1_plot = results_df[results_df["Experiment"] == "1_freezing"].sort_values("n_freeze")
axes[0].bar(range(len(exp1_plot)), exp1_plot["UAR"].values,
            color=["#2ecc71" if nf == BEST_FREEZE else "#bdc3c7" for nf in exp1_plot["n_freeze"]],
            edgecolor="white")
axes[0].set_xticks(range(len(exp1_plot)))
axes[0].set_xticklabels([f"freeze {nf}" for nf in exp1_plot["n_freeze"]], rotation=45)
axes[0].set_ylabel("Test UAR")
axes[0].set_title("Freezing Depth")
axes[0].axhline(y=0.539, color="gray", linestyle="--", alpha=0.5)
for i, v in enumerate(exp1_plot["UAR"].values):
    axes[0].text(i, v + 0.01, f"{v:.1%}", ha="center", fontsize=8, fontweight="bold")

exp2_plot = results_df[results_df["Experiment"] == "2_loss"]
axes[1].bar(range(len(exp2_plot)), exp2_plot["UAR"].values,
            color=["#2ecc71" if l == BEST_LOSS else "#bdc3c7" for l in exp2_plot["Loss"]],
            edgecolor="white")
axes[1].set_xticks(range(len(exp2_plot)))
axes[1].set_xticklabels(exp2_plot["Loss"].values)
axes[1].set_ylabel("Test UAR")
axes[1].set_title(f"Loss Function (freeze {BEST_FREEZE})")
axes[1].axhline(y=0.539, color="gray", linestyle="--", alpha=0.5)
for i, v in enumerate(exp2_plot["UAR"].values):
    axes[1].text(i, v + 0.01, f"{v:.1%}", ha="center", fontsize=8, fontweight="bold")

exp3_plot = results_df[results_df["Experiment"] == "3_augmentation"]
aug_labels = ["ON" if a else "OFF" for a in exp3_plot["Augment"]]
axes[2].bar(range(len(exp3_plot)), exp3_plot["UAR"].values,
            color=["#2ecc71" if a == BEST_AUG else "#bdc3c7" for a in exp3_plot["Augment"]],
            edgecolor="white")
axes[2].set_xticks(range(len(exp3_plot)))
axes[2].set_xticklabels(aug_labels)
axes[2].set_ylabel("Test UAR")
axes[2].set_title(f"Augmentation (freeze {BEST_FREEZE}, {BEST_LOSS})")
axes[2].axhline(y=0.539, color="gray", linestyle="--", alpha=0.5)
for i, v in enumerate(exp3_plot["UAR"].values):
    axes[2].text(i, v + 0.01, f"{v:.1%}", ha="center", fontsize=8, fontweight="bold")

fig.suptitle("Ablation Factor Impact on Test UAR", fontsize=15, y=1.03)
fig.tight_layout()
plt.show()

In [ ]:
display(Markdown("### Key Takeaways"))

best_m = best_overall["metrics"]
best_cfg = best_overall["cfg"]

exp1_uars = results_df[results_df["Experiment"] == "1_freezing"]["UAR"]

takeaway = f"""
| | Value |
|---|---|
| **Zero-shot baseline** | 53.9% UAR |
| **Best fine-tuned** | {best_m['UAR']:.1%} UAR |
| **Improvement** | {best_m['UAR'] - 0.539:+.1%} absolute |
| **Best config** | freeze={best_cfg['n_freeze']}, {best_cfg['loss']}, aug={best_cfg['augment']}, conf={best_cfg['confidence_weighting']} |
| **Macro F1** | {best_m['Macro F1']:.1%} |
| **Freezing range** | UAR from {exp1_uars.min():.1%} (worst) to {exp1_uars.max():.1%} (best) |
| **Epochs trained** | {best_overall['epochs_trained']} |

**Observations:**
1. Freezing depth is the most impactful factor (UAR range: {exp1_uars.max() - exp1_uars.min():.1%}).
2. Fine-tuning yields {best_m['UAR'] - 0.539:+.1%} absolute UAR improvement over zero-shot.
3. Best per-class recalls: {', '.join(f"{e}={best_m[f'{e.capitalize()} R']:.0%}" for e in EMOTIONS)}.
"""
display(Markdown(takeaway))

---
## 14. Save All Results to Google Drive

In [ ]:
import json

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
ABLATION_DIR = DRIVE_PROJECT / "results" / f"ablation_{timestamp}"
ABLATION_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = ABLATION_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

results_df.to_csv(ABLATION_DIR / "ablation_results.csv", index=False)

torch.save(best_overall["best_state"], ABLATION_DIR / "best_model.pt")

with open(ABLATION_DIR / "best_config.json", "w") as f:
    json.dump({
        **best_overall["cfg"],
        "metrics": best_overall["metrics"],
        "best_val_uar": best_overall["best_val_uar"],
        "epochs_trained": best_overall["epochs_trained"],
    }, f, indent=2, default=str)

for r in ALL_RESULTS:
    cfg_key = (r["cfg"]["n_freeze"], r["cfg"]["loss"],
               r["cfg"]["augment"], r["cfg"]["confidence_weighting"])
    run_info = {
        "run_name": r["run_name"],
        "cfg": r["cfg"],
        "metrics": r["metrics"],
        "best_val_uar": r["best_val_uar"],
        "epochs_trained": r["epochs_trained"],
    }
    with open(ABLATION_DIR / f"{r['run_name']}.json", "w") as f:
        json.dump(run_info, f, indent=2, default=str)

plot_confusion_matrix(
    best_overall["test_labels"], best_overall["test_preds"],
    f"Best: {best_overall['run_name']}",
    save_path=str(FIG_DIR / "cm_best"),
)
plot_training_curves(
    best_overall["hist"],
    title=f"Best: {best_overall['run_name']}",
    save_path=str(FIG_DIR / "curves_best"),
)

print(f"{'=' * 60}")
print(f"  ALL ABLATION RESULTS SAVED")
print(f"{'=' * 60}")
print(f"  {ABLATION_DIR}")
print(f"    ablation_results.csv")
print(f"    best_model.pt")
print(f"    best_config.json")
print(f"    figures/cm_best.png/.pdf")
print(f"    figures/curves_best.png/.pdf")
print(f"    exp*.json (per-run details)")

---
## 15. Download Results (Optional)

In [ ]:
import shutil
from google.colab import files

zip_name = f"ablation_{timestamp}"
zip_path = f"/content/{zip_name}"
shutil.make_archive(zip_path, "zip", ABLATION_DIR)
files.download(f"{zip_path}.zip")
print(f"Downloading {zip_name}.zip")